# Phase 2 — Reproducibility Experiment

Establish whether the current v0.1.x ML baseline can reproduce the same dataset split, preprocessing output, predictions, and metrics under the committed environment.

**Experiment only:** this notebook does not modify production pipeline code. Characterize first; harden after evidence is recorded.

### Baseline under test
- Dataset: `data/raw/phisingData.csv`
- Target: `Result` (`-1 → 0` during transformation)
- Split: `test_size=0.2`, `random_state=42`, no stratification
- Preprocessing: `KNNImputer(n_neighbors=3, weights="uniform")`
- Winning model: Random Forest
- Parameters: `n_estimators=128`, `criterion="gini"`, `bootstrap=True`, `max_depth=None`, `max_features="sqrt"`


## Questions

1. Is the dataset fingerprint stable?
2. Is the current train/test split reproducible?
3. Is preprocessing reproducible?
4. Is the current Random Forest reproducible without an explicit model seed?
5. Does `random_state=42` remove any model-level variation?
6. What should be hardened in v0.2.x?


In [1]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
DATASET_PATH = ROOT / "data" / "raw" / "phisingData.csv"
REPORT_PATH = ROOT / "evaluation" / "reproducibility_baseline.json"
TARGET = "Result"
TEST_SIZE = 0.2
SPLIT_SEED = 42
MODEL_SEED = 42
MODEL_PARAMS = {
    "n_estimators": 128,
    "criterion": "gini",
    "bootstrap": True,
    "max_depth": None,
    "max_features": "sqrt",
}

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
print("Repository root:", ROOT)
print("Dataset:", DATASET_PATH)


Repository root: e:\Projects\Network security log triage agent\notebooks
Dataset: e:\Projects\Network security log triage agent\notebooks\data\raw\phisingData.csv


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def dataframe_fingerprint(df: pd.DataFrame) -> str:
    payload = pd.util.hash_pandas_object(df, index=True).to_numpy().tobytes()
    columns = "|".join(map(str, df.columns)).encode("utf-8")
    return hashlib.sha256(columns + payload).hexdigest()

def array_fingerprint(array: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()

dataset_sha256 = sha256_file(DATASET_PATH)
print("Dataset SHA256:", dataset_sha256)


Dataset SHA256: a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995


In [3]:
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
print(json.dumps(environment, indent=2))


{
  "python": "3.11.13",
  "platform": "Windows-10-10.0.26200-SP0",
  "machine": "AMD64",
  "numpy": "2.4.6",
  "pandas": "2.3.3",
  "scikit_learn": "1.9.0"
}


In [4]:
df = pd.read_csv(DATASET_PATH)
print("Shape:", df.shape)
print("Target counts:", df[TARGET].value_counts().to_dict())
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Dataframe fingerprint:", dataframe_fingerprint(df))
assert df.shape == (11055, 31)


Shape: (11055, 31)
Target counts: {1: 6157, -1: 4898}
Exact duplicate rows: 5206
Dataframe fingerprint: 44da7c443a4861a2910cadbc57b3a1faea0581567078ab1f6bce0d8ffc09b5ca


## 1. Split reproducibility

The production ingestion component currently calls `train_test_split` with `test_size` from configuration and `random_state=42`, without stratification. We reproduce that behavior exactly and compare complete dataframe fingerprints, not only row counts.


In [5]:
def make_current_split(dataframe):
    return train_test_split(
        dataframe,
        test_size=TEST_SIZE,
        random_state=SPLIT_SEED,
    )

train_a, test_a = make_current_split(df)
train_b, test_b = make_current_split(df)

split_result = {
    "train_shape": list(train_a.shape),
    "test_shape": list(test_a.shape),
    "train_fingerprint_a": dataframe_fingerprint(train_a),
    "train_fingerprint_b": dataframe_fingerprint(train_b),
    "test_fingerprint_a": dataframe_fingerprint(test_a),
    "test_fingerprint_b": dataframe_fingerprint(test_b),
    "train_identical": train_a.equals(train_b),
    "test_identical": test_a.equals(test_b),
}
print(json.dumps(split_result, indent=2))
assert split_result["train_identical"] and split_result["test_identical"]


{
  "train_shape": [
    8844,
    31
  ],
  "test_shape": [
    2211,
    31
  ],
  "train_fingerprint_a": "6d918e242fbd86da0d868499efa38d1c9067a03bf35882f8cebfdae5737f1b85",
  "train_fingerprint_b": "6d918e242fbd86da0d868499efa38d1c9067a03bf35882f8cebfdae5737f1b85",
  "test_fingerprint_a": "be6c5f2a7a341e25c5718b31f181950b05f1ccf1e4f99eaea84c218511ed3ee1",
  "test_fingerprint_b": "be6c5f2a7a341e25c5718b31f181950b05f1ccf1e4f99eaea84c218511ed3ee1",
  "train_identical": true,
  "test_identical": true
}


## 2. Preprocessing reproducibility

The current transformation fits `KNNImputer(n_neighbors=3, weights="uniform")` on training features and applies it to both training and test features. The target is transformed separately (`-1 → 0`).


In [6]:
def preprocess_current(train_df, test_df):
    x_train = train_df.drop(columns=[TARGET])
    y_train = train_df[TARGET].replace(-1, 0).to_numpy()
    x_test = test_df.drop(columns=[TARGET])
    y_test = test_df[TARGET].replace(-1, 0).to_numpy()

    imputer = KNNImputer(n_neighbors=3, weights="uniform")
    x_train_t = imputer.fit_transform(x_train)
    x_test_t = imputer.transform(x_test)
    return x_train_t, y_train, x_test_t, y_test

x_train_a, y_train_a, x_test_a, y_test_a = preprocess_current(train_a, test_a)
x_train_b, y_train_b, x_test_b, y_test_b = preprocess_current(train_b, test_b)

preprocessing_result = {
    "train_shape": list(x_train_a.shape),
    "test_shape": list(x_test_a.shape),
    "train_identical": np.array_equal(x_train_a, x_train_b),
    "test_identical": np.array_equal(x_test_a, x_test_b),
    "train_fingerprint_a": array_fingerprint(x_train_a),
    "train_fingerprint_b": array_fingerprint(x_train_b),
    "test_fingerprint_a": array_fingerprint(x_test_a),
    "test_fingerprint_b": array_fingerprint(x_test_b),
}
print(json.dumps(preprocessing_result, indent=2))
assert preprocessing_result["train_identical"] and preprocessing_result["test_identical"]


{
  "train_shape": [
    8844,
    30
  ],
  "test_shape": [
    2211,
    30
  ],
  "train_identical": true,
  "test_identical": true,
  "train_fingerprint_a": "e9cd829e2f31163c9ebaff55da309ecb47b05a757123920a24d66ede257fde7d",
  "train_fingerprint_b": "e9cd829e2f31163c9ebaff55da309ecb47b05a757123920a24d66ede257fde7d",
  "test_fingerprint_a": "8a927a6b968ad941bfeccfa0d9ad1dcca30d570be38d823218794114c51b7914",
  "test_fingerprint_b": "8a927a6b968ad941bfeccfa0d9ad1dcca30d570be38d823218794114c51b7914"
}


## 3. Current Random Forest reproducibility

The baseline trainer does not explicitly set a model-level `random_state`. This test intentionally preserves that behavior. If repeated fits differ, record it as evidence for v0.2.x hardening rather than changing production code inside the experiment.


In [7]:
def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

def fit_current_rf(x_train, y_train, x_test, y_test):
    model = RandomForestClassifier(**MODEL_PARAMS)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    return model, pred, evaluate_predictions(y_test, pred)

model_a, pred_a, metrics_a = fit_current_rf(x_train_a, y_train_a, x_test_a, y_test_a)
model_b, pred_b, metrics_b = fit_current_rf(x_train_a, y_train_a, x_test_a, y_test_a)

current_model_result = {
    "metrics_run_a": metrics_a,
    "metrics_run_b": metrics_b,
    "predictions_identical": np.array_equal(pred_a, pred_b),
    "prediction_fingerprint_a": array_fingerprint(pred_a),
    "prediction_fingerprint_b": array_fingerprint(pred_b),
}
print(json.dumps(current_model_result, indent=2))


{
  "metrics_run_a": {
    "accuracy": 0.9669832654907282,
    "f1": 0.9711576451995259,
    "precision": 0.963166144200627,
    "recall": 0.9792828685258964,
    "confusion_matrix": [
      [
        909,
        47
      ],
      [
        26,
        1229
      ]
    ]
  },
  "metrics_run_b": {
    "accuracy": 0.9674355495251018,
    "f1": 0.9715415019762846,
    "precision": 0.9639215686274509,
    "recall": 0.9792828685258964,
    "confusion_matrix": [
      [
        910,
        46
      ],
      [
        26,
        1229
      ]
    ]
  },
  "predictions_identical": false,
  "prediction_fingerprint_a": "1e399f8978a47e7a44d697aa813fe058e606679eacd5fd4c640e4e1ee375491d",
  "prediction_fingerprint_b": "191f7cca18f3bcc8519594c663b90937920ea5ca8f594e8e57e24aa0c7e67ffc"
}


## 4. Explicit model-seed control

Repeat the same model with `random_state=42`. This is a controlled experiment only; it establishes whether an explicit estimator seed is sufficient to stabilize model outputs.


In [8]:
def fit_seeded_rf(x_train, y_train, x_test, y_test):
    model = RandomForestClassifier(**MODEL_PARAMS, random_state=MODEL_SEED)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    return model, pred, evaluate_predictions(y_test, pred)

seeded_a, seeded_pred_a, seeded_metrics_a = fit_seeded_rf(x_train_a, y_train_a, x_test_a, y_test_a)
seeded_b, seeded_pred_b, seeded_metrics_b = fit_seeded_rf(x_train_a, y_train_a, x_test_a, y_test_a)

seeded_model_result = {
    "metrics_run_a": seeded_metrics_a,
    "metrics_run_b": seeded_metrics_b,
    "predictions_identical": np.array_equal(seeded_pred_a, seeded_pred_b),
    "prediction_fingerprint_a": array_fingerprint(seeded_pred_a),
    "prediction_fingerprint_b": array_fingerprint(seeded_pred_b),
}
print(json.dumps(seeded_model_result, indent=2))
assert seeded_model_result["predictions_identical"]


{
  "metrics_run_a": {
    "accuracy": 0.968340117593849,
    "f1": 0.9723320158102767,
    "precision": 0.9647058823529412,
    "recall": 0.9800796812749004,
    "confusion_matrix": [
      [
        911,
        45
      ],
      [
        25,
        1230
      ]
    ]
  },
  "metrics_run_b": {
    "accuracy": 0.968340117593849,
    "f1": 0.9723320158102767,
    "precision": 0.9647058823529412,
    "recall": 0.9800796812749004,
    "confusion_matrix": [
      [
        911,
        45
      ],
      [
        25,
        1230
      ]
    ]
  },
  "predictions_identical": true,
  "prediction_fingerprint_a": "5f5d3b04cf3fa47a075a88aba4afc421c0fb8202b21d646318a723e200968d66",
  "prediction_fingerprint_b": "5f5d3b04cf3fa47a075a88aba4afc421c0fb8202b21d646318a723e200968d66"
}


## 5. Compare with the recorded v0.1.x baseline

Recorded baseline test metrics:

- F1: `0.9707740916271722`
- Precision: `0.9624119028974158`
- Recall: `0.9792828685258964`

This comparison is informational because the historical result came from the full model-selection/training pipeline, while this notebook isolates the final Random Forest configuration.


In [9]:
RECORDED_BASELINE = {
    "f1": 0.9707740916271722,
    "precision": 0.9624119028974158,
    "recall": 0.9792828685258964,
}
comparison = pd.DataFrame([
    {"metric": m, "recorded_baseline": RECORDED_BASELINE[m], "seeded_run": seeded_metrics_a[m]}
    for m in ("f1", "precision", "recall")
])
comparison["absolute_difference"] = (comparison["seeded_run"] - comparison["recorded_baseline"]).abs()
comparison


,metric,recorded_baseline,seeded_run,absolute_difference
0,f1,0.970774,0.972332,0.001558
1,precision,0.962412,0.964706,0.002294
2,recall,0.979283,0.980080,0.000797


## 6. Save the reproducibility evidence

The JSON report is an experiment artifact. It should be reviewed before any production hardening changes are made.


In [10]:
reproducibility_report = {
    "experiment": {
        "name": "phase_2_reproducibility_baseline",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "completed",
    },
    "dataset": {
        "path": DATASET_PATH.as_posix(),
        "sha256": dataset_sha256,
        "shape": list(df.shape),
        "dataframe_fingerprint": dataframe_fingerprint(df),
        "target": TARGET,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    },
    "split": {
        "test_size": TEST_SIZE,
        "random_state": SPLIT_SEED,
        "stratified": False,
        **split_result,
    },
    "preprocessing": {
        "transformer": "KNNImputer",
        "n_neighbors": 3,
        "weights": "uniform",
        **preprocessing_result,
    },
    "model": {
        "algorithm": "RandomForestClassifier",
        "params": MODEL_PARAMS,
        "current_configuration": current_model_result,
        "explicit_seed_configuration": {
            "random_state": MODEL_SEED,
            **seeded_model_result,
        },
    },
    "recorded_v0_1_x_baseline": RECORDED_BASELINE,
    "environment": environment,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(reproducibility_report, indent=2), encoding="utf-8")
print("Wrote:", REPORT_PATH)


Wrote: e:\Projects\Network security log triage agent\notebooks\evaluation\reproducibility_baseline.json


## 7. Interpretation and next action

### Expected interpretation

- Identical split fingerprints → the current `random_state=42` split is reproducible.
- Identical preprocessing fingerprints → the current KNN-imputation stage is reproducible.
- Different unseeded Random Forest predictions/metrics → model-level stochasticity exists.
- Identical seeded Random Forest predictions/metrics → an explicit estimator seed is a justified v0.2.x hardening candidate.

### Important limitation

This experiment does **not** yet prove that the complete production pipeline is bit-for-bit reproducible. Timestamped artifacts, external MongoDB ingestion, MLflow/DagsHub tracking, model-selection logic, persistence, and other pipeline behavior should be evaluated separately.

After this notebook is executed and the evidence is recorded, proceed to **Phase 3: Duplicate Investigation / Leakage-Safe Evaluation**. Do not add `drop_duplicates()` or other data-quality changes before the baseline evidence is preserved.
